# Assignment 9 — Object Detection in Authorized CCTV Footage (YOLO)
**Aim:** Detect objects in user-provided *authorized* CCTV footage with a pretrained Ultralytics YOLO model; report confidence scores, object counts, an annotated sample frame, a count graph, and processed-video instructions.

> **Ethics/authorization:** Use only footage you own or are explicitly authorized to process. Do not upload private third-party video to shared notebooks. Videos and downloaded model weights (`*.pt`, `*.mp4/.avi/.mov`) are **never committed** to this repo.


## 1. Library imports

In [1]:
import os, sys, glob
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import cv2
    print("cv2:", cv2.__version__)
except Exception as e:
    cv2 = None
    print("cv2 not available:", e)

try:
    from ultralytics import YOLO
    HAS_YOLO = True
    print("ultralytics YOLO available")
except Exception as e:
    HAS_YOLO = False
    print("ultralytics not available (demo/mock mode will be used):", e)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)

cv2: 5.0.0
ultralytics not available (demo/mock mode will be used): No module named 'ultralytics'


## 2. Video input — upload prompt (Colab) + path configuration
In Google Colab this cell prompts for an upload. Locally, set `VIDEO_PATH` to your authorized file.


In [2]:
# Colab upload prompt (harmless no-op outside Colab)
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    print("If no video uploaded yet, run the uploader below (Colab only).")
    # Uncomment to upload in Colab:
    # uploaded = files.upload()
    # print("Uploaded:", list(uploaded.keys()))

# >>> USER SETTING: point to your authorized CCTV clip (never commit it) <<<
VIDEO_PATH = os.environ.get("CCTV_VIDEO_PATH", "")  # e.g. "/content/my_cctv.mp4"
SAVE_PROCESSED_VIDEO = False  # set True in Colab to write an annotated mp4 (not committed)
CONF_THRESHOLD = 0.25
print(f"VIDEO_PATH={VIDEO_PATH!r}, SAVE_PROCESSED_VIDEO={SAVE_PROCESSED_VIDEO}, CONF={CONF_THRESHOLD}")

Running in Colab: False
VIDEO_PATH='', SAVE_PROCESSED_VIDEO=False, CONF=0.25


## 3. Dataset / video-input validation

In [3]:
def validate_video(path):
    info = {"exists": False, "readable": False, "frames": 0, "fps": 0.0, "w": 0, "h": 0}
    if path and os.path.isfile(path):
        info["exists"] = True
        if cv2 is not None:
            cap = cv2.VideoCapture(path)
            if cap.isOpened():
                info["readable"] = True
                info["fps"] = float(cap.get(cv2.CAP_PROP_FPS) or 0)
                info["frames"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
                info["w"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
                info["h"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
            cap.release()
    return info

vinfo = validate_video(VIDEO_PATH)
print("Video validation:", vinfo)
USE_REAL_VIDEO = bool(vinfo["exists"] and vinfo["readable"] and HAS_YOLO and cv2 is not None)
print("Will run real YOLO inference:", USE_REAL_VIDEO)
print("If False -> reproducible synthetic CCTV-style demo runs instead (so the notebook always executes).")

Video validation: {'exists': False, 'readable': False, 'frames': 0, 'fps': 0.0, 'w': 0, 'h': 0}
Will run real YOLO inference: False
If False -> reproducible synthetic CCTV-style demo runs instead (so the notebook always executes).


## 4. Model implementation — pretrained YOLO (or reproducible mock demo)

In [4]:
import pandas as pd

detections = []   # rows: frame_idx, class_name, confidence, x1,y1,x2,y2
sample_frame = None
sample_dets = []

if USE_REAL_VIDEO:
    # Real path: pretrained YOLOv8n (weights auto-downloaded to Colab cache, never committed)
    model = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(VIDEO_PATH)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        res = model.predict(frame, conf=CONF_THRESHOLD, verbose=False)[0]
        if idx == 0:
            sample_frame = frame.copy()
        for b in res.boxes:
            cls = model.names[int(b.cls[0])]
            conf = float(b.conf[0])
            x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
            detections.append({"frame": idx, "class": cls, "confidence": conf,
                               "x1": x1, "y1": y1, "x2": x2, "y2": y2})
            if idx == 0:
                sample_dets.append((cls, conf, (x1, y1, x2, y2)))
        idx += 1
    cap.release()
    print(f"Real YOLO: {idx} frames, {len(detections)} detections")
else:
    # Reproducible synthetic demo: 60 frames, 1-4 mock objects/frame
    rng = np.random.default_rng(RANDOM_STATE)
    classes = ["person", "car", "chair"]
    H, W = 360, 640
    sample_frame = np.full((H, W, 3), 60, dtype=np.uint8)
    for f in range(60):
        n = int(rng.integers(1, 5))
        for _ in range(n):
            cls = str(rng.choice(classes, p=[0.6, 0.25, 0.15]))
            conf = float(np.clip(rng.normal(0.72, 0.15), 0.30, 0.98))
            x1, y1 = float(rng.uniform(0, W-90)), float(rng.uniform(0, H-90))
            x2, y2 = x1 + float(rng.uniform(40, 90)), y1 + float(rng.uniform(50, 110))
            detections.append({"frame": f, "class": cls, "confidence": conf,
                               "x1": x1, "y1": y1, "x2": x2, "y2": y2})
            if f == 10:
                sample_dets.append((cls, conf, (x1, y1, x2, y2)))
                if cv2 is not None:
                    cv2.rectangle(sample_frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
    print(f"Synthetic demo: 60 frames, {len(detections)} mock detections (seed=42)")

det_df = pd.DataFrame(detections)
print(det_df.head(10).to_string(index=False))
det_df.to_csv("outputs/detections.csv", index=False)
print("Saved: outputs/detections.csv")

Synthetic demo: 60 frames, 143 mock detections (seed=42)
 frame  class  confidence         x1         y1         x2         y2
     0 person    0.832568 383.552416  25.427884 472.333534 121.096266
     1    car    0.717480 247.712266 100.115467 334.050515 188.747374
     1    car    0.790126 124.981297 149.737892 168.172160 249.395763
     1    car    0.692271 194.989282 262.088467 279.645339 358.791476
     1 person    0.799846  24.092071  41.658163  98.244519 136.343892
     2 person    0.597934 258.255696  51.157267 304.751771 129.699563
     2 person    0.817589 240.433555 224.823113 315.446810 293.565111
     3    car    0.754824 213.113108  77.848588 287.237884 136.233737
     3 person    0.730137 432.808408 179.509731 508.066677 276.353473
     3 person    0.624168  76.888349  30.923120 150.308497 109.188892
Saved: outputs/detections.csv


## 5. Evaluation — confidence scores + object counts

In [5]:
print("Detections:", len(det_df))
print("\nPer-class counts:"); print(det_df["class"].value_counts())
print("\nPer-class mean confidence:"); print(det_df.groupby("class")["confidence"].mean().round(4))
print("\nOverall confidence describe:"); print(det_df["confidence"].describe().round(4))
counts = det_df.groupby(["frame", "class"]).size().unstack(fill_value=0)
counts.to_csv("outputs/per_frame_counts.csv")
mean_conf = det_df.groupby("class")["confidence"].mean().round(4)
print("\nSaved: outputs/per_frame_counts.csv")

Detections: 143

Per-class counts:
class
person    93
car       27
chair     23
Name: count, dtype: int64

Per-class mean confidence:
class
car       0.7606
chair     0.7126
person    0.7157
Name: confidence, dtype: float64

Overall confidence describe:
count    143.0000
mean       0.7237
std        0.1392
min        0.3000
25%        0.6381
50%        0.7436
75%        0.8159
max        0.9800
Name: confidence, dtype: float64

Saved: outputs/per_frame_counts.csv


## 6. Tables and graphs — annotated frame + object-count graph

In [6]:
# Annotated sample frame
fig, ax = plt.subplots(figsize=(9.6, 5.4))
if cv2 is not None:
    rgb = cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB)
else:
    rgb = sample_frame[:, :, ::-1] if sample_frame.shape[2] == 3 else sample_frame
ax.imshow(rgb)
for cls, conf, (x1, y1, x2, y2) in sample_dets[:8]:
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
    ax.text(x1, max(y1-6, 4), f"{cls} {conf:.2f}", color="black", fontsize=8,
            bbox=dict(facecolor="lime", alpha=0.85, pad=1, edgecolor="none"))
ax.set_title(f"Annotated sample frame ({len(sample_dets)} boxes shown)")
ax.axis("off"); plt.tight_layout()
plt.savefig("outputs/annotated_frame.png", dpi=150); plt.close()
print("Saved: outputs/annotated_frame.png")

# Object-count graph (detections per frame + per-class trend)
per_frame_total = det_df.groupby("frame").size()
plt.figure(figsize=(10, 4.5))
plt.plot(per_frame_total.index, per_frame_total.values, marker="o", ms=3, label="Total detections")
for c in sorted(det_df["class"].unique()):
    s = det_df[det_df["class"] == c].groupby("frame").size()
    plt.plot(s.index, s.values, marker=".", ms=4, label=c)
plt.xlabel("Frame index"); plt.ylabel("Object count")
plt.title("Object counts per frame")
plt.legend(); plt.tight_layout()
plt.savefig("outputs/object_counts.png", dpi=150); plt.close()
print("Saved: outputs/object_counts.png")

Saved: outputs/annotated_frame.png


Saved: outputs/object_counts.png


## 7. Processed-video output instructions (video itself is never committed)
- **Colab (real video):** set `VIDEO_PATH` to your uploaded clip, `pip install ultralytics opencv-python`, then run all cells. To also save an annotated video, set `SAVE_PROCESSED_VIDEO=True` — the notebook writes e.g. `/content/annotated_output.mp4` (keep it out of git; download via `files.download(...)`).
- **Reference snippet:**
  ```python
  # pip install ultralytics opencv-python
  # from ultralytics import YOLO
  # model = YOLO("yolov8n.pt")  # pretrained, cached — do NOT commit *.pt
  # results = model.track(source=VIDEO_PATH, conf=0.25, save=True)  # saves annotated video
  ```
- **This repo commit contains only:** notebook + README + PNG/CSV in `outputs/` (no `*.mp4`, no `*.pt`).


## 8. Result and conclusion
- Video input validated; YOLO (real) or seeded mock (demo) detections produced with confidence scores and counts.
- `outputs/detections.csv`, `outputs/per_frame_counts.csv`, `outputs/annotated_frame.png`, `outputs/object_counts.png` written at runtime.
- Replace `VIDEO_PATH` with authorized footage in Colab for real inference; weights/videos stay out of version control.
